# 3. The five observational estimators

**Goal:** estimate the top-3 effect from algorithm-ranked searches only, where the ranker put
better hotels on top, and see how much of the confounding each method removes.

Inverse propensity weighting and AIPW are first written by hand in a few lines of NumPy, so the
mechanics are visible, then run through the library code.

## Setup

Imports and configuration. Every threshold and setting comes from `config.yaml`. Set `SYNTHETIC = True` to run on the synthetic file built by `python -m causallab prepare --synthetic`.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json
import numpy as np
import pandas as pd
from causallab.config import load_config, load_spec, results_dir

cfg = load_config()
SYNTHETIC = False   # True = run on the synthetic Expedia-like file (no Kaggle data needed)
pd.set_option("display.width", 140)

## The data and the reference specification

**What:** the Q1 rows from the observational arm, and the adjustment set fixed in Phase 1.

**Why:** the adjustment set holds variables that drive both the ranker's placement and clicks:
hotel attributes, price and its rank on the page, promotion, query match and search context.

**Expect:** about 1.2 million rows and 44 adjustment variables.

In [2]:
from causallab import data, estimators as E, features
spec = load_spec(cfg)
df = features.q1_rows(data.load_arm("observational", cfg, SYNTHETIC))
cov = spec["adjustment_set"]
y, t, g = df.click_bool.to_numpy(float), df.top3.to_numpy(int), df.srch_id.to_numpy()
print(f"{len(df):,} rows, {len(cov)} adjustment variables")

1,218,303 rows, 44 adjustment variables


## 1. The naive comparison

**What:** click rate of top-3 hotels minus click rate of positions 4-10.

**Why:** it shows how big the confounding is: it mixes the effect of position with the ranker's
choice of better hotels for the top.

**Expect:** about 10 points, roughly double the benchmark.

In [3]:
naive = E.naive(y, t, g)
print(f"naive: {naive.estimate*100:.2f} pp  [{naive.ci_low*100:.2f}, {naive.ci_high*100:.2f}]")

naive: 9.77 pp  [9.64, 9.89]


## 2. The propensity score, cross-fitted by search

**What:** P(top-3 | covariates), from LightGBM. Cross-fitting means each row's score comes from a
model trained on the *other* folds, and folds are groups of whole searches.

**Why:** a score from a model that saw the row overfits; a fold that splits a search leaks
information between training and test rows.

**Expect:** most scores well inside 0 and 1 (on the real sample the largest is about 0.98), so
trimming at 0.01 and 0.99 removes almost no rows.

In [4]:
folds = E.group_folds(g, cfg["models"]["cv_folds"], cfg["seed"])
e, _ = E.crossfit_propensity(df[cov], t, folds, cfg, cfg["seed"])
print(pd.Series(e).describe().round(3))

count    1218303.000
mean           0.346
std            0.162
min            0.051
25%            0.225
50%            0.315
75%            0.431
max            0.982
dtype: float64


## 3. Inverse propensity weighting by hand

**What:** weight each treated row by 1/e and each control row by 1/(1-e), normalise the weights,
compare weighted means.

**Why:** a top-3 hotel that the ranker was *unlikely* to put on top looks like the controls, so it
gets a large weight. Weighting builds a pseudo-population where placement no longer depends on the
covariates.

**Expect:** a big drop from the naive estimate.

In [5]:
e = np.clip(e, 0.01, 0.99)  # never divide by 0; the library trims rows outside [0.01, 0.99] instead
w1, w0 = t / e, (1 - t) / (1 - e)
ipw_by_hand = (w1 * y).sum() / w1.sum() - (w0 * y).sum() / w0.sum()
print(f"IPW by hand: {ipw_by_hand*100:.2f} pp")

IPW by hand: 6.96 pp


## 4. AIPW (doubly robust) by hand

**What:** predict each row's click probability under top-3 (m1) and under control (m0), then
correct the prediction errors with inverse weights:

`psi = m1 - m0 + t*(y - m1)/e - (1-t)*(y - m0)/(1-e)`; the estimate is the mean of `psi`.

**Why:** consistent if *either* the outcome model or the propensity model is right. The same `psi`
gives the standard error: sum it within each search, then square (a clustered standard error).

**Expect:** close to the IPW estimate, with a slightly narrower interval.

In [6]:
m1, m0 = E.crossfit_outcome(df[cov], y, t, folds, cfg, cfg["seed"])
psi = m1 - m0 + t * (y - m1) / e - (1 - t) * (y - m0) / (1 - e)
ate = psi.mean()
se = E.cluster_se(psi - ate, g)
print(f"AIPW by hand: {ate*100:.2f} pp, clustered SE {se*100:.2f}")

AIPW by hand: 6.61 pp, clustered SE 0.06


## 5. All five estimators through the library

**What:** naive, regression adjustment (linear probability model, statsmodels, clustered errors),
IPW, AIPW and double machine learning (EconML), all on the same trimmed rows.

**Why:** DML regresses the outcome residual on the treatment residual. When the effect varies
across hotels it estimates a *variance-weighted* average, not the average effect, which is why AIPW
is the primary estimate.

**Expect:** naive > regression > IPW ≈ AIPW ≈ DML; a few minutes on a laptop.

In [7]:
out = E.estimate_all(df, "top3", "click_bool", cov, "srch_id", cfg)
table = pd.DataFrame({k: {"estimate_pp": v.estimate*100, "ci_low": v.ci_low*100, "ci_high": v.ci_high*100}
                      for k, v in out["estimates"].items()}).T.round(2)
table

,estimate_pp,ci_low,ci_high
naive_untrimmed,9.77,9.64,9.89
naive,9.77,9.64,9.89
regression,8.04,7.92,8.17
ipw,6.96,6.83,7.09
aipw,6.61,6.49,6.73
dml,6.53,6.40,6.65


## How much of the confounding did adjustment remove?

**What:** compare with the benchmark computed in notebook 02.

**Why:** this is the within-study comparison: the gap between the observational estimate and the
experiment is the headline result, whatever it is.

**Expect:** most, but not all, of the naive gap removed.

In [8]:
p = results_dir(cfg) / ("benchmark_synthetic.json" if SYNTHETIC else "benchmark.json")
if not p.exists():  # notebook 02 (or `python -m causallab benchmark`) writes it; build it here if not
    from causallab.benchmark import run_untrimmed
    run_untrimmed(cfg, SYNTHETIC)
bench = json.loads(p.read_text(encoding="utf-8"))["benchmark"]
a, n, b = out["estimates"]["aipw"].estimate, out["estimates"]["naive"].estimate, bench["estimate"]
print(f"benchmark {b*100:.2f} pp | naive {n*100:.2f} | AIPW {a*100:.2f}")
print(f"share of the naive gap removed: {(n - a) / (n - b):.0%}")

benchmark 5.25 pp | naive 9.77 | AIPW 6.61
share of the naive gap removed: 70%


## Sensitivity check: a wider comparison group

**What:** the spec's robustness check: compare the top 3 with every non-reserved position from 4 down,
not just 4-10, with the same specification, trimming and gap test. The benchmark module recomputes the
benchmark for the same window.

**Why:** the 4-10 window is a choice. Widening it changes both sides (lower hotels are worse, and clicks
keep falling with position), so the question is whether the verdict holds, not whether the number stays
the same.

**Expect:** more rows trimmed than in the main run, because hotels far down the page rarely have any
chance of a top-3 slot; the overlap line says whether the comparison is usable.

In [9]:
# `python -m causallab estimate --all-positions` saves this check; reuse the file, or run it if missing
from causallab.sensitivity_window import run_all_positions
p = results_dir(cfg) / ("sensitivity_all_positions_synthetic.json" if SYNTHETIC else "sensitivity_all_positions.json")
sw = json.loads(p.read_text(encoding="utf-8")) if p.exists() else run_all_positions(cfg, SYNTHETIC)
print(f"overlap {'pass' if sw['overlap']['passed'] else 'FAIL'} ({sw['overlap']['trimmed_share']:.1%} trimmed), "
      f"benchmark {sw['benchmark']['estimate'] * 100:.2f} pp")
pd.DataFrame.from_dict({k: {"estimate_pp": v["estimate"] * 100, "gap_pp": sw["gap"][k]["gap"] * 100,
                            "verdict": sw["gap"][k]["verdict"]} for k, v in sw["estimates"].items()}, orient="index").round(2)

overlap pass (0.6% trimmed), benchmark 7.59 pp


,estimate_pp,gap_pp,verdict
naive_untrimmed,12.95,5.36,mismatch
naive,12.94,5.34,mismatch
regression,10.23,2.63,mismatch
ipw,7.91,0.32,match
aipw,7.33,-0.27,match
dml,8.05,0.46,match
